# VarunaNet -- Phase 3 real training (Kaggle GPU)

Runs the real U-Net training loop against the actual Sen1Floods11 chips on a free Kaggle GPU, as an interim path while SLURM cluster access is pending.

**Before running:** in the notebook's right-hand Settings panel, set **Accelerator -> GPU T4 x2 (or P100)** and turn **Internet -> On** (needed to clone the repo and download the public dataset). Then run every cell top to bottom.

This does *not* pin the exact `torch`/`torchvision` versions `training/environment.yml` pins for the A100 cluster -- those were chosen for a specific driver/CUDA target, and forcing them here risks a CUDA/driver mismatch against whatever Kaggle's current GPU image actually has. Safer to trust Kaggle's own pre-installed, already GPU-matched `torch` build and only add the packages this project actually needs on top of it. The cluster run (once SLURM access is confirmed) is what uses the exact pinned environment for the official result.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/ski-akash/VarunaNet.git
%cd VarunaNet

In [ ]:
# Only the project-specific packages -- torch/torchvision/numpy come
# from Kaggle's own image (see the note above on why those aren't pinned
# here).
!pip install -q segmentation-models-pytorch==0.5.0 hydra-core==1.3.5 omegaconf==2.3.1 \
    wandb==0.28.2 rasterio==1.5.1 pysheds==0.5 affine==3.0.0 scikit-image==0.26.0 scikit-learn==1.9.0

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "No GPU visible -- check the Settings panel (Accelerator = GPU T4 x2 or P100)"

## Fetch the real data

Both of these hit public, no-auth sources (a public GCS bucket for Sen1Floods11's hand-labeled subset, and Copernicus DEM on AWS Open Data for terrain) -- same scripts used to build the data locally, no Kaggle Dataset upload needed. `data/sen1floods11_normalization_stats.json` is already committed to the repo (computed once on the real training split), so it doesn't need recomputing here.

In [ ]:
!python -m data.download_sen1floods11

In [ ]:
!python -m data.fetch_dem

## Train 3 seeds

Spec section 4.2's exit criterion wants mean ± std over 3 seeds. `epochs=100` is a generous ceiling -- early stopping (`early_stopping_patience=5`, the config default) is what's actually expected to end each run, once val IoU stops improving. `wandb.mode=offline` needs no API key or network call to W&B; the run data is still written locally under each seed's `wandb/` dir if you want to `wandb sync` it later.

Each seed gets its own `checkpoint_dir` -- sharing one would let the seeds silently overwrite each other's `best.pt`.

In [ ]:
!python -m training.train dataset=sen1floods11 device=cuda epochs=100 seed=1 \
    checkpoint_dir=checkpoints/seed_1 wandb.mode=offline

In [ ]:
!python -m training.train dataset=sen1floods11 device=cuda epochs=100 seed=2 \
    checkpoint_dir=checkpoints/seed_2 wandb.mode=offline

In [ ]:
!python -m training.train dataset=sen1floods11 device=cuda epochs=100 seed=3 \
    checkpoint_dir=checkpoints/seed_3 wandb.mode=offline

## Score each seed's best checkpoint against the official *test* split

This is the comparison the exit criterion actually needs -- val IoU (printed during training above) was only ever used for early stopping and picking `best.pt`, never for the baseline comparison itself. Uses the same `benchmarks/metrics.py` scoring the classical baselines (Otsu, Otsu+HAND, Random Forest) were measured with.

In [ ]:
import os
import numpy as np
from hydra import compose, initialize_config_dir

from training.evaluate_test import evaluate_test

CONFIG_DIR = os.path.abspath("training/conf")

seed_ious = {}
for seed in (1, 2, 3):
    with initialize_config_dir(config_dir=CONFIG_DIR, version_base=None):
        cfg = compose(
            config_name="evaluate_test",
            overrides=[f"checkpoint=checkpoints/seed_{seed}/best.pt", "device=cuda"],
        )
    summary = evaluate_test(cfg)
    seed_ious[seed] = summary.mean_iou
    print(f"seed {seed}: test mean_iou={summary.mean_iou:.4f} median_iou={summary.median_iou:.4f} "
          f"mean_f1={summary.mean_f1:.4f} mean_precision={summary.mean_precision:.4f} mean_recall={summary.mean_recall:.4f}")

ious = np.array(list(seed_ious.values()))
print()
print(f"U-Net test mean IoU = {ious.mean():.4f} +/- {ious.std(ddof=1):.4f} (n=3 seeds)")
print()
print("Classical baselines (official test split, benchmarks/RESULTS.md as of the per-chip-Otsu version):")
print("  Otsu:          mean IoU 0.211")
print("  Otsu + HAND:   mean IoU 0.197")
print("  Random Forest: mean IoU 0.234")
print("(These baseline numbers are mid-revision -- see VarunaNet_Spec.md's pending per-event-Otsu-threshold fix. Re-check benchmarks/RESULTS.md for the current numbers before treating this as the final exit-criterion comparison.)")

## Download the checkpoints

Kaggle notebook sessions are ephemeral -- zip up the checkpoints (and offline W&B run data) so they survive after the session ends. Download `checkpoints.zip` from the notebook's Output/Data pane once this finishes.

In [ ]:
!zip -r -q /kaggle/working/checkpoints.zip checkpoints wandb
print("wrote /kaggle/working/checkpoints.zip")